In [2]:
import pandas as pd
from prophet import Prophet
from tqdm import tqdm
import numpy as np

# === CONFIGURATION === #
FILENAME = "datasets/dataset_with_targets.csv"
FUTURE_PERIODS = 306  # Months: July 2025 to December 2050
STATIC_FEATURES = [
    "city", "geopotential_height", "high_vegetation_cover", "high_vegetation_type",
    "lake_cover", "land_sea_mask", "low_vegetation_cover", "low_vegetation_type",
    "soil_type", "target_latitude", "target_longitude", "grid_latitude", "grid_longitude"
]
TARGET_COLUMNS = [
    "monsoon_intensity", "climate_change", "siltation",
    "agricultural_practices", "landslide_risks"
]

DEFAULT_NOISE = 0.005  # Very small noise

SHRINK_TOWARD_MEAN = 0.5  # How much to shrink predictions toward historical mean
APPLY_SMOOTHING = True
SMOOTHING_WINDOW = 3

# === LOAD DATA === #
df = pd.read_csv(FILENAME)
df['date'] = pd.to_datetime(df['date'])

# Save original column order, excluding targets
original_order = [col for col in df.columns if col not in TARGET_COLUMNS]

# Determine dynamic features
excluded_cols = set(STATIC_FEATURES + TARGET_COLUMNS + ['date'])
dynamic_features = [col for col in df.columns if col not in excluded_cols]

# Output container
synthetic_data_all = []

# Process each city
for city in tqdm(df['city'].unique(), desc="Processing cities"):
    city_df = df[df['city'] == city].copy()
    static_values = city_df.iloc[0][STATIC_FEATURES].to_dict()
    
    forecasted_features = pd.DataFrame()

    for feature in dynamic_features:
        ts = city_df[['date', feature]].rename(columns={'date': 'ds', feature: 'y'})
        
        if ts['y'].nunique() < 2:
            continue  # Skip features with no variance

        model = Prophet(
            yearly_seasonality=True,
            daily_seasonality=False,
            weekly_seasonality=False,
            seasonality_mode='additive'
        )

        # Comment out biannual if needed
        # model.add_seasonality(name='biannual', period=182.5, fourier_order=3)

        try:
            model.fit(ts)
        except Exception as e:
            print(f"⚠️ Failed to fit Prophet for {feature} in {city}: {e}")
            continue

        future = model.make_future_dataframe(periods=FUTURE_PERIODS, freq='MS')
        forecast = model.predict(future)
        predicted = forecast[['ds', 'yhat']].tail(FUTURE_PERIODS).rename(columns={'ds': 'date', 'yhat': feature})

        # Minimal noise
        std_dev = ts['y'].std()
        noise = np.random.normal(0, DEFAULT_NOISE * std_dev, size=len(predicted))
        predicted[feature] += noise

        # Shrink toward historical mean
        historical_mean = ts['y'].mean()
        predicted[feature] = (
            SHRINK_TOWARD_MEAN * historical_mean +
            (1 - SHRINK_TOWARD_MEAN) * predicted[feature]
        )

        # Hard clipping: stay within 5th–90th percentile
        lower_bound = ts['y'].quantile(0.05)
        upper_bound = ts['y'].quantile(0.90)
        predicted[feature] = predicted[feature].clip(lower=lower_bound, upper=upper_bound)

        # Optional: Smooth results
        if APPLY_SMOOTHING:
            predicted[feature] = predicted[feature].rolling(window=SMOOTHING_WINDOW, min_periods=1).mean()

        # Merge
        if forecasted_features.empty:
            forecasted_features = predicted
        else:
            forecasted_features = forecasted_features.merge(predicted, on='date')

    # Add static features
    for key, val in static_values.items():
        forecasted_features[key] = val

    # Add city column explicitly if missing
    if 'city' not in forecasted_features.columns:
        forecasted_features['city'] = city

    # Reorder columns to match original file (excluding targets)
    final_columns = [col for col in original_order if col != 'date']
    forecasted_features = forecasted_features[['date'] + final_columns]

    synthetic_data_all.append(forecasted_features)

# Concatenate all cities
final_synthetic_df = pd.concat(synthetic_data_all, ignore_index=True)

# Save output
final_synthetic_df.to_csv("datasets/synthetic_features_small_conservative.csv", index=False)
print("✅ Conservative synthetic dataset saved as 'synthetic_features_small_conservative.csv'")


Processing cities:   0%|          | 0/3 [00:00<?, ?it/s]07:57:36 - cmdstanpy - INFO - Chain [1] start processing
07:57:36 - cmdstanpy - INFO - Chain [1] done processing
07:57:36 - cmdstanpy - INFO - Chain [1] start processing
07:57:36 - cmdstanpy - INFO - Chain [1] done processing
07:57:36 - cmdstanpy - INFO - Chain [1] start processing
07:57:36 - cmdstanpy - INFO - Chain [1] done processing
07:57:37 - cmdstanpy - INFO - Chain [1] start processing
07:57:37 - cmdstanpy - INFO - Chain [1] done processing
07:57:37 - cmdstanpy - INFO - Chain [1] start processing
07:57:37 - cmdstanpy - INFO - Chain [1] done processing
07:57:37 - cmdstanpy - INFO - Chain [1] start processing
07:57:37 - cmdstanpy - INFO - Chain [1] done processing
07:57:37 - cmdstanpy - INFO - Chain [1] start processing
07:57:37 - cmdstanpy - INFO - Chain [1] done processing
07:57:37 - cmdstanpy - INFO - Chain [1] start processing
07:57:37 - cmdstanpy - INFO - Chain [1] done processing
07:57:38 - cmdstanpy - INFO - Chain [1] 

✅ Conservative synthetic dataset saved as 'synthetic_features_small_conservative.csv'


### Full Synthesis

In [3]:
import pandas as pd
from prophet import Prophet
from tqdm import tqdm
import numpy as np

# === CONFIGURATION === #
FILENAME = "datasets/dataset_with_targets.csv"
FUTURE_PERIODS = 306  # Months: July 2025 to December 2050
STATIC_FEATURES = [
    "city", "geopotential_height", "high_vegetation_cover", "high_vegetation_type",
    "lake_cover", "land_sea_mask", "low_vegetation_cover", "low_vegetation_type",
    "soil_type", "target_latitude", "target_longitude", "grid_latitude", "grid_longitude"
]
TARGET_COLUMNS = [
    "monsoon_intensity", "climate_change", "siltation",
    "agricultural_practices", "landslide_risks"
]

DEFAULT_NOISE = 0.01  # Small noise
SHRINK_TOWARD_MEAN = 0.3  # Gently shrink to historical mean
APPLY_SMOOTHING = True
SMOOTHING_WINDOW = 3

# === LOAD DATA === #
df = pd.read_csv(FILENAME)
df['date'] = pd.to_datetime(df['date'])

# Save original column order
original_order = df.columns.tolist()

# Dynamic features now includes both predictors + target columns
excluded_cols = set(STATIC_FEATURES + ['date'])
dynamic_features = [col for col in df.columns if col not in excluded_cols]

# Output container
synthetic_data_all = []

# Process each city
for city in tqdm(df['city'].unique(), desc="Processing cities"):
    city_df = df[df['city'] == city].copy()
    static_values = city_df.iloc[0][STATIC_FEATURES].to_dict()
    
    forecasted_features = pd.DataFrame()

    for feature in dynamic_features:
        ts = city_df[['date', feature]].rename(columns={'date': 'ds', feature: 'y'})
        
        if ts['y'].nunique() < 2:
            continue  # Skip constant features

        model = Prophet(
            yearly_seasonality=True,
            daily_seasonality=False,
            weekly_seasonality=False,
            seasonality_mode='additive'
        )

        try:
            model.fit(ts)
        except Exception as e:
            print(f"⚠️ Failed to fit Prophet for {feature} in {city}: {e}")
            continue

        future = model.make_future_dataframe(periods=FUTURE_PERIODS, freq='MS')
        forecast = model.predict(future)
        predicted = forecast[['ds', 'yhat']].tail(FUTURE_PERIODS).rename(columns={'ds': 'date', 'yhat': feature})

        # Add small Gaussian noise
        std_dev = ts['y'].std()
        noise = np.random.normal(0, DEFAULT_NOISE * std_dev, size=len(predicted))
        predicted[feature] += noise

        # Shrink forecast gently toward historical mean
        hist_mean = ts['y'].mean()
        predicted[feature] = (
            SHRINK_TOWARD_MEAN * hist_mean +
            (1 - SHRINK_TOWARD_MEAN) * predicted[feature]
        )

        # Clip to realistic bounds (5th–90th percentile)
        q5 = ts['y'].quantile(0.05)
        q90 = ts['y'].quantile(0.90)
        predicted[feature] = predicted[feature].clip(lower=q5, upper=q90)

        # Optional smoothing
        if APPLY_SMOOTHING:
            predicted[feature] = predicted[feature].rolling(window=SMOOTHING_WINDOW, min_periods=1).mean()

        # Merge feature
        if forecasted_features.empty:
            forecasted_features = predicted
        else:
            forecasted_features = forecasted_features.merge(predicted, on='date')

    # Add static features
    for key, val in static_values.items():
        forecasted_features[key] = val

    # Add city column explicitly if missing
    if 'city' not in forecasted_features.columns:
        forecasted_features['city'] = city

    # Reorder to match original file (now includes targets)
    reordered = ['date'] + [col for col in original_order if col != 'date']
    forecasted_features = forecasted_features[[col for col in reordered if col in forecasted_features.columns]]

    synthetic_data_all.append(forecasted_features)

# Concatenate all cities
final_synthetic_df = pd.concat(synthetic_data_all, ignore_index=True)

# Save output
final_synthetic_df.to_csv("datasets/synthetic_all_features_incl_targets.csv", index=False)
print("✅ Full synthetic dataset with targets saved as 'synthetic_all_features_incl_targets.csv'")


Processing cities:   0%|          | 0/3 [00:00<?, ?it/s]08:06:02 - cmdstanpy - INFO - Chain [1] start processing
08:06:02 - cmdstanpy - INFO - Chain [1] done processing
08:06:02 - cmdstanpy - INFO - Chain [1] start processing
08:06:02 - cmdstanpy - INFO - Chain [1] done processing
08:06:02 - cmdstanpy - INFO - Chain [1] start processing
08:06:02 - cmdstanpy - INFO - Chain [1] done processing
08:06:02 - cmdstanpy - INFO - Chain [1] start processing
08:06:02 - cmdstanpy - INFO - Chain [1] done processing
08:06:02 - cmdstanpy - INFO - Chain [1] start processing
08:06:02 - cmdstanpy - INFO - Chain [1] done processing
08:06:03 - cmdstanpy - INFO - Chain [1] start processing
08:06:03 - cmdstanpy - INFO - Chain [1] done processing
08:06:03 - cmdstanpy - INFO - Chain [1] start processing
08:06:03 - cmdstanpy - INFO - Chain [1] done processing
08:06:03 - cmdstanpy - INFO - Chain [1] start processing
08:06:03 - cmdstanpy - INFO - Chain [1] done processing
08:06:03 - cmdstanpy - INFO - Chain [1] 

✅ Full synthetic dataset with targets saved as 'synthetic_all_features_incl_targets.csv'


In [4]:
import pandas as pd
from prophet import Prophet
from tqdm import tqdm
import numpy as np

# === CONFIGURATION === #
FILENAME = "datasets/dataset_with_targets.csv"
FUTURE_PERIODS = 306  # July 2025 to December 2050
STATIC_FEATURES = [
    "city", "geopotential_height", "high_vegetation_cover", "high_vegetation_type",
    "lake_cover", "land_sea_mask", "low_vegetation_cover", "low_vegetation_type",
    "soil_type", "target_latitude", "target_longitude", "grid_latitude", "grid_longitude"
]
TARGET_COLUMNS = [
    "monsoon_intensity", "climate_change", "siltation",
    "agricultural_practices", "landslide_risks"
]

DEFAULT_NOISE = 0.01  # Small noise
SHRINK_TOWARD_MEAN = 0.5  # STRONGER shrink to historical mean
APPLY_SMOOTHING = True
SMOOTHING_WINDOW = 3

# City-specific shrink factors
CITY_SHRINK_OVERRIDES = {
    "Kampala": 0.7,  # Heavily shrink toward mean
    # Add others as needed
}

# === LOAD DATA === #
df = pd.read_csv(FILENAME)
df['date'] = pd.to_datetime(df['date'])

# Save original column order
original_order = df.columns.tolist()

# Dynamic features = predictors + target variables
excluded_cols = set(STATIC_FEATURES + ['date'])
dynamic_features = [col for col in df.columns if col not in excluded_cols]

# Output container
synthetic_data_all = []

# Process each city
for city in tqdm(df['city'].unique(), desc="Processing cities"):
    city_df = df[df['city'] == city].copy()
    static_values = city_df.iloc[0][STATIC_FEATURES].to_dict()

    forecasted_features = pd.DataFrame()

    for feature in dynamic_features:
        ts = city_df[['date', feature]].rename(columns={'date': 'ds', feature: 'y'})
        
        if ts['y'].nunique() < 2:
            continue  # Skip constant series

        model = Prophet(
            yearly_seasonality=True,
            daily_seasonality=False,
            weekly_seasonality=False,
            seasonality_mode='additive'
        )

        try:
            model.fit(ts)
        except Exception as e:
            print(f"⚠️ Prophet failed for {feature} in {city}: {e}")
            continue

        future = model.make_future_dataframe(periods=FUTURE_PERIODS, freq='MS')
        forecast = model.predict(future)
        predicted = forecast[['ds', 'yhat']].tail(FUTURE_PERIODS).rename(columns={'ds': 'date', 'yhat': feature})

        # Add small noise
        std_dev = ts['y'].std()
        noise = np.random.normal(0, DEFAULT_NOISE * std_dev, size=len(predicted))
        predicted[feature] += noise

        # Shrink toward mean — city-specific if defined
        hist_mean = ts['y'].mean()
        shrink_factor = CITY_SHRINK_OVERRIDES.get(city, SHRINK_TOWARD_MEAN)
        predicted[feature] = (
            shrink_factor * hist_mean +
            (1 - shrink_factor) * predicted[feature]
        )

        # Clip to safe range (10th–80th percentile)
        q10 = ts['y'].quantile(0.10)
        q80 = ts['y'].quantile(0.80)
        predicted[feature] = predicted[feature].clip(lower=q10, upper=q80)

        # Optional smoothing
        if APPLY_SMOOTHING:
            predicted[feature] = predicted[feature].rolling(window=SMOOTHING_WINDOW, min_periods=1).mean()

        # Merge feature
        if forecasted_features.empty:
            forecasted_features = predicted
        else:
            forecasted_features = forecasted_features.merge(predicted, on='date')

    # Add static features
    for key, val in static_values.items():
        forecasted_features[key] = val

    # Add city column if missing
    if 'city' not in forecasted_features.columns:
        forecasted_features['city'] = city

    # Reorder to match original file
    reordered = ['date'] + [col for col in original_order if col != 'date']
    forecasted_features = forecasted_features[[col for col in reordered if col in forecasted_features.columns]]

    synthetic_data_all.append(forecasted_features)

# Concatenate all cities
final_synthetic_df = pd.concat(synthetic_data_all, ignore_index=True)

# Save output
final_synthetic_df.to_csv("datasets/synthetic_all_features_incl_targets.csv", index=False)
print("✅ Conservative synthetic dataset with targets saved: 'synthetic_all_features_incl_targets.csv'")


Processing cities:   0%|          | 0/3 [00:00<?, ?it/s]08:11:40 - cmdstanpy - INFO - Chain [1] start processing
08:11:40 - cmdstanpy - INFO - Chain [1] done processing
08:11:40 - cmdstanpy - INFO - Chain [1] start processing
08:11:40 - cmdstanpy - INFO - Chain [1] done processing
08:11:40 - cmdstanpy - INFO - Chain [1] start processing
08:11:40 - cmdstanpy - INFO - Chain [1] done processing
08:11:40 - cmdstanpy - INFO - Chain [1] start processing
08:11:40 - cmdstanpy - INFO - Chain [1] done processing
08:11:40 - cmdstanpy - INFO - Chain [1] start processing
08:11:40 - cmdstanpy - INFO - Chain [1] done processing
08:11:40 - cmdstanpy - INFO - Chain [1] start processing
08:11:41 - cmdstanpy - INFO - Chain [1] done processing
08:11:41 - cmdstanpy - INFO - Chain [1] start processing
08:11:41 - cmdstanpy - INFO - Chain [1] done processing
08:11:41 - cmdstanpy - INFO - Chain [1] start processing
08:11:41 - cmdstanpy - INFO - Chain [1] done processing
08:11:41 - cmdstanpy - INFO - Chain [1] 

✅ Conservative synthetic dataset with targets saved: 'synthetic_all_features_incl_targets.csv'
